# Classificação multiclasse com SageMaker - Dataset Wine

O algoritmo a seguir implementa um modelo de Machine Learning para classificar vinhos usando o algoritmo XGBoost, executado na plataforma AWS SageMaker.

A ideia é fornecer ao modelo informações sobre a composição química de cada vinho e pedir que ele preveja a qual classe esse vinho pertence.

O dataset é multiclasse

O Amazon S3 é um serviço para armazenar arquivos na nuvem. Um bucket funciona como um contêiner para guardar arquivos e os resultados gerados pelo modelo

In [ ]:
bucket='INSIRA O VALOR DO SEU BUCKET AQUI'

In [ ]:
import warnings, io, os
warnings.simplefilter('ignore')
import pandas as pd
import numpy as np
import boto3
import sagemaker
from sagemaker.image_uris import retrieve
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Etapa 1: Carregar e preparar os dados

O XGBoost do SageMaker espera CSV **sem cabeçalho**, com o alvo na **primeira coluna**. Também remove a coluna `id`.

In [ ]:
df = pd.read_csv('train.csv')
print(df.shape)
print(df['target'].value_counts().sort_index())

# removendo a coluna id
df = df.drop(columns=['id'])
# tratando valores ausentes, no caso substituindo esses valores NaN por uma mediana calculada
df = df.fillna(df.median())

# colocando o alvo na primeira coluna
cols = ['target'] + [c for c in df.columns if c != 'target']
df = df[cols]

# identifica quais classes tem
classes = sorted(df['target'].unique())
# conta qnts classes tem
num_class = len(classes)
print('Classes:', classes)

# Verificando se as classes partem do valor 0, sem pular nenhum número
assert classes == list(range(num_class))
df.head()

In [ ]:
# Os dados serão divididos em 3 partes
# 70% treino - ensinar o modelo a reconhecer padrões
# 15% validação - acompanhar o desempenho durante o treinamento e apoiar ajustes
# 15% teste - avaliar o modelo com dados separados do processo de aprendizagem

#train_test_split divide a tabela em duas partes (70% train e 30% test e validate)
train, test_and_validate = train_test_split(
  df,
  test_size=0.3,
  random_state=42,
  stratify=df['target'] # procura preservar a proporção das classes em cada conjunto
)

# Pega os 30% de test e validate e divide em aprox. 15% para cada
test, validate = train_test_split(
    test_and_validate,
    test_size=0.5,
    random_state=42,
    stratify=test_and_validate['target'])

print(train.shape, validate.shape, test.shape)

# var que definem nomes de pasta-base e dos arquivos CSV (ajudam a organizar os dados no S3)
prefix = 'lab3-wine'
train_file = 'wine_train.csv'
test_file = 'wine_test.csv'
validate_file = 'wine_validate.csv'

# Criar conexão com S3
s3_resource = boto3.Session().resource('s3')

# Função para enviar as tabelas para o S3
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

upload_s3_csv(train_file, 'train', train)
upload_s3_csv(test_file, 'test', test)
upload_s3_csv(validate_file, 'validate', validate)

# Etapa 2: Treinar o modelo (XGBoost multiclasse)

Diferenças em relação ao laboratório: `objective` passa a ser `multi:softmax`, com `num_class` informado, e a métrica `auc` é trocada por `merror` (taxa de erro multiclasse).

In [ ]:

# O SageMaker executa algoritmos em ambientes chamados container
container = retrieve(
    'xgboost', # nome do algoritmo
    boto3.Session().region_name, #região da AWS
    '1.0-1'# versão
)

# Configurando parãmetros do modelo (Como o XGBoost vai aprender?)
hyperparams = {
    'num_round': '42', # qnt de rodadas de treinamento
    'eval_metric': 'merror', # métrica que calcula a proporção de classificações incorretas
    'objective': 'multi:softmax', # define uma tarefa de classificação multiclasse e retorna a classe prevista
    'num_class': str(num_class) # definem quantas classes existem
}

# Criar o estimador
s3_output_location = 's3://{}/{}/output/'.format(
    bucket, prefix
)

# configurar o treinamento na nuvem
xgb_model = sagemaker.estimator.Estimator(
    container,
    sagemaker.get_execution_role(), # função de execução que fornece as permissões necessárias ao SageMaker
    instance_count=1, # utiliza uma instância computacional
    instance_type='ml.m4.xlarge', # tipo de máquina solicitado
    output_path=s3_output_location, # local do S3 para guardar os resultados do treinamento
    hyperparameters=hyperparams, # parâmetros definidos anteriormente.
    sagemaker_session=sagemaker.Session() # sessão de comunicação com o SageMaker
)

# Informar onde estão os dados
train_channel = sagemaker.inputs.TrainingInput(
    's3://{}/{}/train/'.format(bucket, prefix),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    's3://{}/{}/validate/'.format(bucket, prefix),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

# iniciar o treinamento
xgb_model.fit(inputs=data_channels, logs=False)

# Etapa 3: Predições nos dados de teste (Batch Transform)
Depois de treinar o modelo, chegou o momento de testar se ele consegue classificar vinhos que não utilizou para aprender

In [ ]:
# test sem a coluna 0 (target)
batch_X = test.iloc[:, 1:]


# salva as caracterísiticas dos vinhos de teste em um arquivo CSV sem cabeçalho e o envia ao S3
batch_X_file = 'batch-in.csv'
upload_s3_csv(batch_X_file, 'batch-in', batch_X)

# Configurar a entrada e a saída

# onde serão armazenadas as previsões
batch_output = 's3://{}/{}/batch-out/'.format(
    bucket, prefix
)

# onde está o arquivo com os atributos que serão classificados
batch_input = 's3://{}/{}/batch-in/{}'.format(
    bucket, prefix, batch_X_file
)

# Criar o processo de predição

# Batch Transform permite fazer previsões em lote
xgb_transformer = xgb_model.transformer(
    instance_count=1,
    instance_type='ml.m4.xlarge',
    strategy='MultiRecord',
    assemble_with='Line',
    output_path=batch_output
)

xgb_transformer.transform(
    data=batch_input,
    data_type='S3Prefix',
    content_type='text/csv',
    split_type='Line'
)

xgb_transformer.wait()

In [ ]:
# Ler o arquivo de saída

s3 = boto3.client('s3')
obj = s3.get_object(Bucket=bucket, Key='{}/batch-out/{}'.format(prefix, 'batch-in.csv.out'))
target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()), names=['class'])

# com multi:softmax a saida ja e a classe prevista (ex.: 1.0); nao ha limiar (threshold)
y_pred = target_predicted['class'].round().astype(int).values
y_true = test.iloc[:, 0].values

print(y_pred[:10])
print(y_true[:10])
print('Acuracia:', (y_pred == y_true).mean())

# Etapa 4: Matriz de confusão (3x3)

In [ ]:
matriz = confusion_matrix(y_true, y_pred, labels=classes)
df_confusion = pd.DataFrame(matriz,
                            index=['Real {}'.format(c) for c in classes],
                            columns=['Previsto {}'.format(c) for c in classes])
print(df_confusion)

colormap = sns.color_palette('BrBG', 10)
plt.figure(figsize=(7, 5))
sns.heatmap(df_confusion, annot=True, fmt='d', cbar=None, cmap=colormap)
plt.title('Matriz de Confusao - Wine (multiclasse)')
plt.ylabel('Classe Real')
plt.xlabel('Classe Prevista')
plt.tight_layout()
plt.show()

# Etapa 5: Precisão e recall de cada classe

Para a classe *k* (linhas = real, colunas = previsto):

- **VP** = valor da diagonal `matriz[k, k]`
- **Precisão** = VP / (VP + FP) = VP / soma da coluna k
- **Recall** = VP / (VP + FN) = VP / soma da linha k

In [ ]:
for i, c in enumerate(classes):
    VP = matriz[i, i]
    soma_coluna = matriz[:, i].sum()
    soma_linha = matriz[i, :].sum()
    FP = soma_coluna - VP
    FN = soma_linha - VP

    precisao = VP / soma_coluna if soma_coluna else 0
    recall = VP / soma_linha if soma_linha else 0

    print(f'Classe {c}: VP={VP}, FP={FP}, FN={FN}')
    print(f'  Precisao = VP / (VP + FP) = {VP} / ({VP} + {FP}) = {precisao:.4f}')
    print(f'  Recall   = VP / (VP + FN) = {VP} / ({VP} + {FN}) = {recall:.4f}')
    print()

In [ ]:
# Conferencia com o scikit-learn: os valores devem bater com os calculos acima
print(classification_report(y_true, y_pred, zero_division=0))